# 05 · GPTQ and AWQ for the Five Core Models

**Project:** Silent Tool-Call Failures: How Quantization Degrades Function Calling in Small LLMs
**Goal:** make 4-bit GPTQ and AWQ checkpoints of every core model with **one recipe** (Update 5), run them, and fill the first main table: FP16 / INT8 / NF4 / GPTQ / AWQ (Update 6).

| | GPTQ | AWQ |
|---|---|---|
| Weights | 4-bit, groups of 128, symmetric, no activation reordering | 4-bit, groups of 128, with zero points |
| Saved as | GPTQ format | AWQ GEMM format |
| Run by vLLM 0.8.5 with | its GPTQ kernels | its AWQ GEMM kernels |

Both are made by `src/quantize.py` with GPTQModel, from the same pinned model revisions as FP16 and the same calibration set for every model: the first 256 English C4 documents (training shard 1) with at least 3,000 characters, each cut to 512 tokens. GPTQModel needs a newer PyTorch than vLLM 0.8.5, so it gets its own Python environment. vLLM cannot use its faster Marlin kernels on a T4 (they need compute capability 8.0); the GPTQ and AWQ kernels it uses instead are slower but give the same kind of 4-bit computation.

Everything else is the same as in Updates 4-5: the 1,390 BFCL `non_live` questions, the prompt, greedy decoding and the scorer. Every run uses one T4.

**Quick check first:** Qwen3-1.7B is quantized with both methods and answers 2 questions per category. If a method fails there, its runs are skipped and the other method still runs.

## How to run on Kaggle

1. kaggle.com → **Create → New Notebook → File → Import Notebook**, then upload this file.
2. In the right panel, under **Settings**: **Accelerator → GPU T4 x2** and **Internet → On**.
3. **Required for Llama:** **Add-ons → Secrets** → add `HF_TOKEN` (a Hugging Face *read* token from huggingface.co → Settings → Access Tokens) and switch it on for this notebook.
4. *Optional, to upload the checkpoints to Hugging Face:* set `HF_UPLOAD_USER` in the settings cell and add a second secret `HF_WRITE_TOKEN` (a token with *write* access). Uploads are private unless `HF_UPLOAD_PRIVATE = False`.
5. Click **Run All** to watch it, or **Save Version → Save & Run All (Commit)** to let it run in the background.

**Time:** probably 4-7 hours. This is a guess: quantizing takes most of it, and the T4 runs 4-bit models with slower kernels. Kaggle stops a session after 12 hours.
**Output:** `update6_results.zip` with the raw answers, scores, run and checkpoint information, logs, `gptq_awq.md` and `main_table.md`.

## 1. Settings
Edit this cell only.

In [ ]:
import os

REPO_URL = "https://github.com/shawalkabirchy/quantization-functional-call"
REPO_REF = "main"             # branch, tag or commit of the code to run

# Grouped by model (GPTQ, then AWQ), so each model is downloaded once; Qwen3-8B last
CONFIGS = [
    "configs/qwen3-1.7b_gptq.yaml", "configs/qwen3-1.7b_awq.yaml",
    "configs/llama-3.2-1b_gptq.yaml", "configs/llama-3.2-1b_awq.yaml",
    "configs/llama-3.2-3b_gptq.yaml", "configs/llama-3.2-3b_awq.yaml",
    "configs/qwen3-4b_gptq.yaml", "configs/qwen3-4b_awq.yaml",
    "configs/qwen3-8b_gptq.yaml", "configs/qwen3-8b_awq.yaml",
]
CHECK_CONFIGS = ["configs/qwen3-1.7b_gptq.yaml", "configs/qwen3-1.7b_awq.yaml"]
CHECK_LIMIT = 2               # questions per category in the quick check
MODE = "free"
REPORT_NAME = "gptq_awq.md"
REPORT_TITLE = "GPTQ and AWQ (Updates 5-6)"
RUN_TIMEOUT_MIN = 150         # a run that takes longer than this is stopped, and the next one starts
QUANTIZE_TIMEOUT_MIN = 120    # the same for making a checkpoint
DELETE_MODEL_AFTER_RUN = True # frees disk space: deletes each checkpoint after its run, each model after its last run

# Optional upload of every checkpoint to Hugging Face (needs the HF_WRITE_TOKEN secret)
HF_UPLOAD_USER = ""           # your Hugging Face user name; empty = no upload
HF_UPLOAD_PRIVATE = True

BFCL_VERSION = "2026.3.23"    # pinned for reproducibility
BITSANDBYTES_VERSION = "0.45.5"  # same vLLM environment as notebook 04
ACCELERATE_VERSION = "1.6.0"
GPTQMODEL_VERSION = "5.7.0"      # the last GPTQModel release that works with transformers 4.x
QUANT_TORCH_VERSION = "2.9.1"
QUANT_TRANSFORMERS_VERSION = "4.57.6"
QUANT_TORCHAO_VERSION = "0.14.1"  # matches torch 2.9
WORK = "/kaggle/working"
REPO = f"{WORK}/repo"                         # code and results (kept as notebook output)
VENV = os.path.expanduser("~/bfcl-venv")      # vLLM + BFCL environment (not kept)
QUANT_VENV = os.path.expanduser("~/quant-venv")  # GPTQModel environment (not kept)
CHECKPOINT_DIR = os.path.expanduser("~/checkpoints")   # GPTQ and AWQ checkpoints (not kept: they are large)
os.environ["CHECKPOINT_DIR"] = CHECKPOINT_DIR           # read by src/quantize.py and src/run_eval.py


def run_id_of(config):
    return os.path.splitext(os.path.basename(config))[0] + f"_{MODE}"

## 2. Check the GPU, internet and Hugging Face tokens

In [ ]:
import json
import shutil
import subprocess
import time
from datetime import date

import requests

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode != 0:
    raise RuntimeError("No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then run again.")
print("GPUs:\n" + smi.stdout)

try:
    requests.get("https://pypi.org/simple/", timeout=15).raise_for_status()
    print("Internet: OK")
except Exception as e:
    raise RuntimeError("No internet. Set Settings -> Internet -> On (needs a phone-verified Kaggle account).") from e

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: loaded from Kaggle Secrets")
except Exception:
    print("HF_TOKEN: NOT SET. Llama 3.2 cannot be downloaded without it (Add-ons -> Secrets).")

if HF_UPLOAD_USER:
    try:
        os.environ["HF_WRITE_TOKEN"] = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
        print(f"HF_WRITE_TOKEN: loaded; checkpoints will be uploaded to {HF_UPLOAD_USER}/... "
              f"({'private' if HF_UPLOAD_PRIVATE else 'public'})")
    except Exception as e:
        raise RuntimeError("HF_UPLOAD_USER is set but the HF_WRITE_TOKEN secret is missing (Add-ons -> Secrets).") from e
else:
    print("Upload to Hugging Face: off (HF_UPLOAD_USER is empty)")

## 3. Get the code and check access to every model
Clones the repository, then downloads one small file of each model at the pinned revision. This finds a missing token or missing Llama approval now, not after an hour of running.

In [ ]:
import yaml
from huggingface_hub import hf_hub_download

shutil.rmtree(REPO, ignore_errors=True)
!git clone --quiet {REPO_URL} {REPO}
!git -C {REPO} checkout --quiet {REPO_REF}
!git -C {REPO} log -1 --format="Code version: %h %s (%ci)"

cfgs = {}
problems = []
for config in CONFIGS:
    with open(f"{REPO}/{config}") as f:
        cfgs[config] = yaml.safe_load(f)
    model, revision = cfgs[config]["model"], str(cfgs[config]["revision"])
    try:
        hf_hub_download(model, "config.json", revision=revision, token=os.environ.get("HF_TOKEN"))
        print(f"  access OK  {model}")
    except Exception as e:
        problems.append(f"{model}: {type(e).__name__}: {str(e)[:200]}")
if problems:
    print("\n".join(problems))
    raise RuntimeError(
        "Some models cannot be downloaded. For Llama: add HF_TOKEN under Add-ons -> Secrets, switch it on for "
        "this notebook, and check that https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct says you have access."
    )

## 4. Install the vLLM environment (about 1-5 minutes)
Exactly the same environment as notebook 04: a separate Python 3.11 environment with `bfcl-eval[oss_eval_vllm]` (vLLM 0.8.5) and the same pins.

In [ ]:
!pip install -q uv
shutil.rmtree(VENV, ignore_errors=True)
!uv venv {VENV} --python 3.11
!uv pip install --python {VENV}/bin/python "bfcl-eval[oss_eval_vllm]=={BFCL_VERSION}" "transformers==4.51.3" "openai>=1.86.0,<2" "fastapi<0.116" "pydantic<2.12" "setuptools>=77.0.3,<80" soundfile "bitsandbytes=={BITSANDBYTES_VERSION}" "accelerate=={ACCELERATE_VERSION}"

CHECK = (
    "import vllm, bfcl_eval, yaml, xgrammar, triton, torch, bitsandbytes, accelerate; "
    "from importlib.metadata import version; "
    "print('vllm', version('vllm'), '| xgrammar', version('xgrammar'), '| torch', version('torch'), "
    "'| bitsandbytes', version('bitsandbytes'), '| accelerate', version('accelerate')); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"
)
r = subprocess.run([f"{VENV}/bin/python", "-c", CHECK], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("Environment check failed (see above). Check the output of the install step.")

freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{VENV}/bin/python"], capture_output=True, text=True)
with open(f"{WORK}/requirements-freeze.txt", "w") as f:
    f.write(freeze.stdout)
print(f"Saved {len(freeze.stdout.splitlines())} package versions to {WORK}/requirements-freeze.txt")

## 5. Install the quantization environment (about 2-5 minutes)
A second Python 3.11 environment with GPTQModel. `BUILD_CUDA_EXT=0` skips compiling GPTQModel's own CUDA kernels: they are only needed to run models in GPTQModel, and here vLLM runs them. GPTQModel saves both formats with its plain PyTorch code.

In [ ]:
shutil.rmtree(QUANT_VENV, ignore_errors=True)
!uv venv {QUANT_VENV} --python 3.11
!BUILD_CUDA_EXT=0 uv pip install --python {QUANT_VENV}/bin/python "gptqmodel=={GPTQMODEL_VERSION}" "torch=={QUANT_TORCH_VERSION}" "transformers=={QUANT_TRANSFORMERS_VERSION}" "torchao=={QUANT_TORCHAO_VERSION}" pyyaml

QUANT_CHECK = (
    "import gptqmodel, torch, transformers, yaml; from importlib.metadata import version; "
    "print('gptqmodel', version('gptqmodel'), '| torch', version('torch'), '| transformers', version('transformers')); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "
    "'| compute capability', torch.cuda.get_device_capability(0))"
)
r = subprocess.run([f"{QUANT_VENV}/bin/python", "-c", QUANT_CHECK], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("Quantization environment check failed (see above). Check the output of the install step.")

freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{QUANT_VENV}/bin/python"], capture_output=True, text=True)
with open(f"{WORK}/requirements-freeze_quant.txt", "w") as f:
    f.write(freeze.stdout)
print(f"Saved {len(freeze.stdout.splitlines())} package versions to {WORK}/requirements-freeze_quant.txt")

## 6. Quick check: does each method quantize, load and answer?
Qwen3-1.7B with GPTQ and with AWQ: make the checkpoint, then answer 2 questions per category. The answers go to `check/`, apart from the real results. A method that fails here is skipped in step 7; its checkpoint is kept and reused there.

In [ ]:
def tail(path, n=25):
    with open(path, errors="replace") as f:
        return "".join(f.readlines()[-n:])


def run_logged(args, log_path, timeout_min, venv=VENV):
    """Runs a script of the repository in an environment; returns its exit code, or None after a timeout."""
    with open(log_path, "w") as log:
        try:
            return subprocess.run([f"{venv}/bin/python", *args], cwd=REPO, stdout=log, stderr=subprocess.STDOUT,
                                  timeout=timeout_min * 60).returncode
        except subprocess.TimeoutExpired:
            print(f"  stopped after {timeout_min} minutes")
            return None


def gpu_memory_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return [int(x) for x in out.split()]


def wait_for_free_gpus(limit_mib=1000, wait_s=120):
    """A finished run should leave the GPUs empty; wait briefly if they are not yet."""
    start = time.time()
    while max(gpu_memory_used_mib()) > limit_mib and time.time() - start < wait_s:
        time.sleep(5)
    used = gpu_memory_used_mib()
    if max(used) > limit_mib:
        print(f"  WARNING: GPU memory still in use before this run: {used} MiB")


CHECK_DIR = f"{WORK}/check"
shutil.rmtree(CHECK_DIR, ignore_errors=True)
works = {}
for config in CHECK_CONFIGS:
    cfg, stem = cfgs[config], os.path.splitext(os.path.basename(config))[0]
    print(f"\n=== check {stem}: started {time.strftime('%H:%M:%S')} UTC", flush=True)
    wait_for_free_gpus()
    ok = True
    if "quantize" in cfg:
        log_path = f"{WORK}/check_quantize_{stem}.log"
        ok = run_logged(["src/quantize.py", "--config", config], log_path, QUANTIZE_TIMEOUT_MIN, QUANT_VENV) == 0
        print("  checkpoint: " + tail(log_path, 1).strip())
        wait_for_free_gpus()
    if ok:
        log_path = f"{WORK}/check_{stem}.log"
        ok = run_logged(["src/run_eval.py", "--config", config, "--mode", MODE, "--limit", str(CHECK_LIMIT),
                         "--results-dir", CHECK_DIR, "--overwrite"], log_path, 30) == 0
    works[cfg["precision"]] = ok
    if not ok:
        print(f"  FAILED. End of {log_path}:\n{tail(log_path, 40)}")
        continue

    run_id = f"{run_id_of(config)}_limit{CHECK_LIMIT}"
    r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"{CHECK_DIR}/raw/{run_id}.jsonl"],
                       cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-1500:])
    with open(f"{CHECK_DIR}/meta/{run_id}.json") as f:
        meta = json.load(f)
    print(f"  weights {meta['memory']['model_weights_gib']} GiB, {meta['output_tokens_per_second']} output tokens/s, "
          f"finish reasons {meta['finish_reasons']}")
    with open(f"{CHECK_DIR}/raw/{run_id}.jsonl") as f:
        for line in list(f)[:3]:
            row = json.loads(line)
            print(f"  {row['id']}: {row['output'][:200]!r}")

print("\nQuick check: " + ", ".join(f"{p.upper()} {'works' if ok else 'FAILED'}" for p, ok in works.items()))
if not all(works.values()):
    print("The runs of a failed method are skipped in step 7; the other method still runs.")

## 7. Quantize, run and score every config (several hours)
One config at a time: `quantize.py` (makes the checkpoint, in the quantization environment), `run_eval.py` (answers), then `score.py` (accuracy). Every step's full log is saved (`quantize_<config>.log`, `run_<run id>.log`). If a config fails, the notebook prints the end of its log and goes on with the next one. With `HF_UPLOAD_USER` set, each checkpoint is uploaded after its run.

In [ ]:
def upload_checkpoint(config, stem):
    """Uploads a finished checkpoint to Hugging Face; returns (repo id, commit hash)."""
    from huggingface_hub import HfApi
    name = cfgs[config]["model"].split("/")[-1]
    repo_id = f"{HF_UPLOAD_USER}/{name}-{cfgs[config]['precision'].upper()}-W4A16-g128"
    api = HfApi(token=os.environ["HF_WRITE_TOKEN"])
    api.create_repo(repo_id, private=HF_UPLOAD_PRIVATE, exist_ok=True)
    commit = api.upload_folder(folder_path=f"{CHECKPOINT_DIR}/{stem}", repo_id=repo_id,
                               commit_message=f"Made by src/quantize.py --config {config} ({REPO_URL})")
    return repo_id, commit.oid


status, minutes, uploads = {}, {}, {}
last_config_of = {cfgs[c]["model"]: c for c in CONFIGS}   # each model is deleted after its last config
for config in CONFIGS:
    cfg, run_id = cfgs[config], run_id_of(config)
    stem = os.path.splitext(os.path.basename(config))[0]
    if not works.get(cfg["precision"], True):
        status[run_id] = "SKIPPED (quick check failed)"
    else:
        print(f"\n=== {run_id} ({cfg['model']}): started {time.strftime('%H:%M:%S')} UTC", flush=True)
        start = time.time()
        ok = True
        if "quantize" in cfg:
            wait_for_free_gpus()
            log_path = f"{WORK}/quantize_{stem}.log"
            ok = run_logged(["src/quantize.py", "--config", config], log_path, QUANTIZE_TIMEOUT_MIN, QUANT_VENV) == 0
            if ok:
                print("  checkpoint: " + tail(log_path, 1).strip())
            else:
                status[run_id] = "QUANTIZE FAILED"
                print(f"  quantization FAILED. End of {log_path}:\n{tail(log_path)}")
        if ok:
            wait_for_free_gpus()
            log_path = f"{WORK}/run_{run_id}.log"
            run_logged(["src/run_eval.py", "--config", config, "--mode", MODE, "--overwrite"], log_path,
                       RUN_TIMEOUT_MIN)
            if not os.path.exists(f"{REPO}/results/meta/{run_id}.json"):
                status[run_id] = "FAILED"
                print(f"  FAILED. End of {log_path}:\n{tail(log_path)}")
            else:
                print("  answers: " + next((l.strip() for l in tail(log_path, 10).splitlines()
                                            if l.startswith("Done:")), ""))
                r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"results/raw/{run_id}.jsonl"],
                                   cwd=REPO, capture_output=True, text=True)
                print(r.stdout[-2000:])
                status[run_id] = "OK" if r.returncode == 0 else "SCORING FAILED"
                if r.returncode != 0:
                    print(r.stderr[-3000:])
        minutes[run_id] = (time.time() - start) / 60
        print(f"  {status[run_id]} after {minutes[run_id]:.1f} min")
        if HF_UPLOAD_USER and os.path.exists(f"{CHECKPOINT_DIR}/{stem}/quantize_info.json"):
            try:
                uploads[stem] = upload_checkpoint(config, stem)
                print(f"  uploaded to https://huggingface.co/{uploads[stem][0]} (commit {uploads[stem][1][:7]})")
            except Exception as e:
                print(f"  upload FAILED: {type(e).__name__}: {str(e)[:300]}")

    if DELETE_MODEL_AFTER_RUN:
        shutil.rmtree(f"{CHECKPOINT_DIR}/{stem}", ignore_errors=True)
        if last_config_of[cfg["model"]] == config:
            shutil.rmtree(os.path.expanduser(f"~/.cache/huggingface/hub/models--{cfg['model'].replace('/', '--')}"),
                          ignore_errors=True)

print("\nSummary:")
for run_id, s in status.items():
    took = f"  ({minutes[run_id]:.1f} min)" if run_id in minutes else ""
    print(f"  {run_id:28s} {s}{took}")
print(f"  total: {sum(minutes.values()) / 60:.1f} hours")
if any(s != "OK" for s in status.values()):
    print("Some runs did not finish. The results of the others are still reported and packaged below.")
if uploads:
    with open(f"{WORK}/hf_uploads.json", "w") as f:
        json.dump({stem: {"repo": repo, "commit": commit} for stem, (repo, commit) in uploads.items()}, f, indent=2)

## 8. Tables: FP16 vs. GPTQ vs. AWQ, and the main table
The FP16 rows are the Update 4 results already in the repository. **Overall** is BFCL's Non-Live Overall Acc (mean of simple, multiple, parallel and parallel-multiple); irrelevance is reported separately. **Parsed** is the share of answers to the AST questions that were valid JSON in the requested format: answers that fail here are syntactic failures. The main table (`results/main_table.md`, made by `src/main_table.py`) has all five precisions; INT8 and NF4 appear there once their results (notebook 04) are in the repository.

In [ ]:
import re

import pandas as pd

COLUMNS = {
    "simple_python": "Python", "simple_java": "Java", "simple_javascript": "JavaScript",
    "multiple": "Multiple", "parallel": "Parallel", "parallel_multiple": "Parallel Multiple",
    "non_live_overall": "Overall", "irrelevance": "Irrelevance",
}
PRECISIONS = ["fp16", "gptq", "awq"]


def table_order(base):
    family = 0 if "qwen" in base else 1
    size = float(re.search(r"(\d+(?:\.\d+)?)b", base).group(1))
    return family, size


bases = sorted({os.path.basename(c).rsplit("_", 1)[0] for c in CONFIGS}, key=table_order)  # e.g. qwen3-1.7b
summary = pd.read_csv(f"{REPO}/results/summary.csv")
acc_rows, info_rows, overall, metas = [], [], {}, {}
for base in bases:
    for precision in PRECISIONS:
        run_id = f"{base}_{precision}_{MODE}"
        s = summary[summary["run_id"] == run_id].set_index("category")
        if s.empty or "non_live_overall" not in s.index:
            continue
        with open(f"{REPO}/results/meta/{run_id}.json") as f:
            meta = metas[run_id] = json.load(f)
        scores = pd.read_json(f"{REPO}/results/scores/{run_id}.jsonl", lines=True)
        ast = scores[scores["category"] != "irrelevance"]
        name, label = meta["model"].split("/")[-1], precision.upper()
        overall.setdefault(name, {})[label] = float(s.loc["non_live_overall", "accuracy"])
        acc_rows.append({"Model": name, "Precision": label,
                         **{col: float(s.loc[c, "accuracy"]) for c, col in COLUMNS.items()}})
        info_rows.append({
            "Model": name,
            "Precision": label,
            "Parsed, AST questions (%)": round(100 * ast["parse_ok"].mean(), 2),
            "Unparsed answers": int((~ast["parse_ok"]).sum()),
            "Cut off at max_tokens": meta["finish_reasons"].get("length", 0),
            "GPUs": meta["config"]["tensor_parallel_size"],
            "Weights per GPU (GiB)": meta["memory"]["model_weights_gib"],
            "Generation (s)": meta["generation_seconds"],
            "Output tokens/s": meta["output_tokens_per_second"],
        })

compare_rows = []
for name, acc in overall.items():
    row = {"Model": name, **{p.upper(): acc.get(p.upper()) for p in PRECISIONS}}
    for p in ["GPTQ", "AWQ"]:
        row[f"{p} - FP16"] = round(acc[p] - acc["FP16"], 2) if p in acc and "FP16" in acc else None
    compare_rows.append(row)

compare = pd.DataFrame(compare_rows)
accuracy = pd.DataFrame(acc_rows)
info = pd.DataFrame(info_rows)
display(compare)
display(accuracy)
display(info)
compare.to_csv(f"{WORK}/gptq_awq_overall.csv", index=False)
accuracy.to_csv(f"{WORK}/gptq_awq_accuracy.csv", index=False)
info.to_csv(f"{WORK}/gptq_awq_run_info.csv", index=False)

if not info.empty:
    quantized = info[info["Precision"] != "FP16"]
    missing = quantized[quantized["Weights per GPU (GiB)"].isna() | quantized["Output tokens/s"].isna()]
    print("Memory and speed recorded for every quantized run." if missing.empty else
          f"WARNING: memory or speed missing for:\n{missing[['Model', 'Precision']]}")

r = subprocess.run([f"{VENV}/bin/python", "src/main_table.py", "--mode", MODE], cwd=REPO, capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else r.stderr[-3000:])

## 9. Write the report and package the results

In [ ]:
import glob


def md_table(df):
    def fmt(v):
        if v is None or (isinstance(v, float) and v != v):
            return ""
        return f"{v:.2f}" if isinstance(v, float) else str(v)
    head = "| " + " | ".join(df.columns) + " |"
    sep = "|" + "|".join("---" if i == 0 else "---:" for i in range(len(df.columns))) + "|"
    return [head, sep] + ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]


first = next((m for m in metas.values() if m["precision"] != "fp16"), None)
lines = [f"# {REPORT_TITLE}", "", f"**Date:** {date.today()}",
         "**Goal:** function-calling accuracy of every core model with 4-bit GPTQ and AWQ checkpoints made with one "
         "recipe, compared with the FP16 baselines of Update 4.", ""]
if first:
    v = first["versions"]
    lines += [
        "## Setup", "",
        f"- **Questions:** BFCL `non_live`, {first['n_questions']} questions per run, bfcl-eval {v['bfcl-eval']}",
        f"- **Prompt format:** `{first['prompt_format']}` (as in Updates 3-4), {MODE} decoding; Qwen3 with thinking off",
        f"- **Decoding:** greedy (temperature 0), seed {first['config']['sampling']['seed']}, "
        f"at most {first['config']['sampling']['max_tokens']} answer tokens",
    ]
    q = first["quantized_checkpoint"]
    c, qv = q["calibration"], q["versions"]
    gptq = next((m["quantized_checkpoint"]["settings"] for m in metas.values() if m["precision"] == "gptq"), None)
    awq = next((m["quantized_checkpoint"]["settings"] for m in metas.values() if m["precision"] == "awq"), None)
    lines += [
        f"- **Inference:** vLLM {v['vllm']} on one {first['gpus'][0]} per run (FP16 Qwen3-8B used two), "
        "with its GPTQ and AWQ GEMM kernels (the faster Marlin kernels need a newer GPU)",
        f"- **Quantization:** GPTQModel {qv['gptqmodel']} (torch {qv['torch']}, transformers {qv['transformers']}), "
        "`src/quantize.py`; the same pinned model revisions as FP16",
        f"- **Calibration (same for every model and both methods):** the first {c['samples']} documents with at least "
        f"{c['min_chars']:,} characters from `{c['dataset']}` `{c['file']}` (revision `{c['revision'][:7]}`), "
        f"each cut to {c['tokens']} tokens",
    ]
    if gptq:
        lines.append(f"- **GPTQ:** {gptq['bits']}-bit weights, groups of {gptq['group_size']}, "
                     f"{'symmetric' if gptq['sym'] else 'asymmetric'}, desc_act={gptq['desc_act']}; "
                     "saved in the GPTQ format")
    if awq:
        lines.append(f"- **AWQ:** {awq['bits']}-bit weights, groups of {awq['group_size']}, "
                     f"{'with' if awq['zero_point'] else 'without'} zero points; saved in the AWQ GEMM format")
    lines += [
        f"- **Software:** Python {v['python']}, torch {v['torch']}, transformers {v['transformers']}",
        f"- **Code:** commit `{(first['code']['commit'] or 'unknown')[:7]}`; model revisions are in `results/meta/`",
        "- **Overall** = BFCL's Non-Live Overall Acc (unweighted mean of simple, multiple, parallel and "
        "parallel-multiple; simple = mean of Python, Java, JavaScript). Irrelevance is reported on its own.",
        "",
        "## Overall accuracy (%)", "",
    ] + md_table(compare) + [
        "", "## Accuracy per category (%)", "",
    ] + md_table(accuracy) + ["", "## Format, length and speed", ""] + md_table(info)
failed = [r for r, s in status.items() if s != "OK"]
lines += ["", "## Notes", ""]
lines += [f"- Did not finish: {', '.join(failed)}" if failed else "- All runs finished."]
if uploads:
    lines += [f"- Checkpoints on Hugging Face: " + ", ".join(f"[{repo}](https://huggingface.co/{repo}) (`{commit[:7]}`)"
                                                         for repo, commit in uploads.values())]
lines += ["- "]
report = "\n".join(lines) + "\n"
with open(f"{WORK}/{REPORT_NAME}", "w") as f:
    f.write(report)
print(report)

pack = f"{WORK}/update6_results"
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(f"{REPO}/results", f"{pack}/results", ignore=shutil.ignore_patterns("bfcl_repro", "figures"))
if os.path.exists(CHECK_DIR):
    shutil.copytree(CHECK_DIR, f"{pack}/check")
for name in [REPORT_NAME, "gptq_awq_overall.csv", "gptq_awq_accuracy.csv", "gptq_awq_run_info.csv",
             "requirements-freeze.txt", "requirements-freeze_quant.txt", "hf_uploads.json"]:
    if os.path.exists(f"{WORK}/{name}"):
        shutil.copy(f"{WORK}/{name}", pack)
for log_path in sorted(glob.glob(f"{WORK}/*.log")):
    shutil.copy(log_path, pack)
archive = shutil.make_archive(pack, "zip", pack)
print("Saved", archive)

## After the run: what goes into the repository

1. Download **`update6_results.zip`** from the notebook's **Output** section (or the saved version's Output tab).
2. Download the notebook with its outputs (**File → Download Notebook**) and replace `notebooks/05_gptq_awq.ipynb` with it.
3. From the zip:
   - `gptq_awq.md` → `results/gptq_awq.md`; write a few sentences under **Notes**
   - `results/meta/`, `results/scores/`, `results/summary.csv` and `results/main_table.md` → the same places in the repository (run `python src/main_table.py` again if the INT8 and NF4 results were added after this run)
   - The raw answers in `results/raw/` stay out of git; keep the zip as a backup.
4. Add the GPU time to `notes/compute_log.md`, the GPTQ/AWQ recipe to `notes/decisions.md`, and the entry to `UPDATES.md`, then commit and push.